In [1]:
import torch
import torch.nn as nn
import torch.functional as F
from torch.utils.data import Dataset, DataLoader

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

import time
import cv2
import numpy as np
import matplotlib.pyplot as plt
import PIL.Image as img
import keyboard
import mouse
import pyautogui
from screeninfo import get_monitors
from tqdm import tqdm
from IPython.display import clear_output

In [2]:
num_hands        = 1
tracking_points  = 21
tracking_dim     = 3

classifier_hidden_dim = 1024
num_action_classes    = 2

lr                       = 0.001
grad_accumulation_seconds = 5

run_device    = torch.device('cuda')
action_labels = ['a', 's']
press_buttons = False

target_monitor_y_res = get_monitors()[1].height
optim = torch.optim.AdamW

In [3]:
class HumanFeedbackOptimizer:
    """
    Specialized optimizer wrapper for real-time human-in-the-loop training.

    The mouse Y position is an external reward signal — it has no grad_fn and
    cannot be included in the autograd graph meaningfully. Instead, this optimizer
    scales gradients directly by the feedback scalar:

        loss = -feedback * scores.mean()

    Mouse at top    (y ≈ 0)         → feedback ≈ +1 → reinforce current predictions
    Mouse at center (y ≈ height/2)  → feedback ≈  0 → dead zone, skip update
    Mouse at bottom (y ≈ height)    → feedback ≈ -1 → punish current predictions
    """

    def __init__(self, wrapped_optimizer, screen_height, dead_zone=0.05):
        self.wrapped = wrapped_optimizer
        self.screen_height = screen_height
        self.dead_zone = dead_zone
        self._accum_count = 0

    def _get_feedback(self):
        """Read mouse Y and return a scalar in [-1, +1]."""
        y = mouse.get_position()[1]
        return 1.0 - 2.0 * (y / self.screen_height)

    def accumulate(self, scores):
        """
        Compute a feedback-scaled backward pass and accumulate gradients.
        Does NOT step the wrapped optimizer — call step() after your accum window.

        Args:
            scores: min-max normalized output from the model — must retain grad_fn.

        Returns:
            float: effective loss value, or None if in the dead zone (no update).
        """
        feedback = self._get_feedback()

        if abs(feedback) < self.dead_zone:
            return None

        loss = -feedback * scores.mean()
        loss.backward()
        self._accum_count += 1
        return loss.item()

    def step(self):
        """
        Average accumulated gradients over the accum window, then step + zero.
        Safe to call even if no accumulation steps occurred.
        """
        if self._accum_count > 0:
            for group in self.wrapped.param_groups:
                for p in group['params']:
                    if p.grad is not None:
                        p.grad.data.div_(self._accum_count)
            self.wrapped.step()

        self.wrapped.zero_grad()
        self._accum_count = 0

    def zero_grad(self):
        self.wrapped.zero_grad()
        self._accum_count = 0

    @property
    def param_groups(self):
        return self.wrapped.param_groups

In [4]:
class CameraWrapper:
    """A simple wrapper for an OpenCV VideoCapture object.
    Provides a clean interface to get frames in RGB numpy format.
    """
    def __init__(self, camera_index: int = 0):
        """Initializes the camera.
        
        Args:
            camera_index: The index of the camera to use (e.g., 0 for the default).
        """
        self.cap = cv2.VideoCapture(camera_index)
        if not self.cap.isOpened():
            raise IOError(f"Cannot open camera at index {camera_index}")

    def __call__(self) -> np.ndarray | None:
        """Reads a single frame from the camera and converts it to RGB.
        
        Returns:
            An RGB numpy array if a frame is successfully read, 
            otherwise None (e.g., on stream end or error).
        """
        ret, frame = self.cap.read()
        
        if ret:
            # Convert from BGR (OpenCV default) to RGB
            return frame
        return None

    def release(self):
        """Releases the camera resource."""
        if self.cap.isOpened():
            self.cap.release()

    def __enter__(self):
        """Enter the runtime context related to this object."""
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        """Exit the runtime context and release the camera."""
        self.release()

In [5]:
class HandTracker:
    """
    MediaPipe Tasks HandLandmarker wrapper that returns torch tensors.

    Output tensor:
      shape: (num_hands, 21, 3)
      coords: normalized x,y in [0,1] (relative to image size), z is model-relative.
    """

    def __init__(
        self,
        model_path: str,
        max_num_hands: int = 2,
        min_hand_detection_confidence: float = 0.5,
        min_hand_presence_confidence: float = 0.5,
        min_tracking_confidence: float = 0.5,
    ):
        self.base_options = python.BaseOptions(model_asset_path=model_path)
        self.options = vision.HandLandmarkerOptions(
            base_options=self.base_options,
            running_mode=vision.RunningMode.VIDEO,
            num_hands=max_num_hands,
            min_hand_detection_confidence=min_hand_detection_confidence,
            min_hand_presence_confidence=min_hand_presence_confidence,
            min_tracking_confidence=min_tracking_confidence,
        )
        self.landmarker = vision.HandLandmarker.create_from_options(self.options)

    def reset(self):
        self.landmarker.close()
        self.landmarker = vision.HandLandmarker.create_from_options(self.options)


    def __call__(self, bgr_frame: np.ndarray, timestamp_ms: int) -> torch.Tensor:
        """
        Args:
            bgr_frame: OpenCV image (H,W,3) uint8 in BGR
            timestamp_ms: monotonically increasing timestamp in milliseconds (required for VIDEO mode)

        Returns:
            torch.FloatTensor of shape (N, 21, 3). If no hands: (0, 21, 3)
        """
        if bgr_frame is None or bgr_frame.ndim != 3 or bgr_frame.shape[2] != 3:
            raise ValueError("bgr_frame must be an HxWx3 BGR image")

        rgb = cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2RGB)

        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = self.landmarker.detect_for_video(mp_image, timestamp_ms=timestamp_ms)

        if not result.hand_landmarks:
            return torch.empty((0, 21, 3), dtype=torch.float32)

        # result.hand_landmarks: list[hands] -> each is list[21] normalized landmarks with x,y,z
        out = np.zeros((len(result.hand_landmarks), 21, 3), dtype=np.float32)
        for h, hand in enumerate(result.hand_landmarks):
            for i, lm in enumerate(hand):
                out[h, i, 0] = lm.x
                out[h, i, 1] = lm.y
                out[h, i, 2] = lm.z

        return torch.from_numpy(out)

    def close(self):
        self.landmarker.close()

In [6]:
_TOPOLOGY = [
    (0, 1), (1, 2), (2, 3), (3, 4),        # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),        # Index
    (0, 9), (9, 10), (10, 11), (11, 12),   # Middle
    (0, 13), (13, 14), (14, 15), (15, 16), # Ring
    (0, 17), (17, 18), (18, 19), (19, 20), # Pinky
]

def to_bone_offsets(tensor: torch.Tensor) -> torch.Tensor:
    """
    Convert absolute MediaPipe coords (N, 21, 3) to parent-relative joint offsets.
    Wrist becomes (0,0,0); each joint is its displacement from its parent.
    Translation-invariant: encodes hand shape only, not position in frame.
    """
    out = tensor.clone()
    for h in range(out.shape[0]):
        out[h, 0] = 0.0
        for parent, child in _TOPOLOGY:
            out[h, child] = tensor[h, child] - tensor[h, parent]
    return out

In [7]:
class HandVisualizer:
    """
    Visualizer for MediaPipe Hand Landmarks returned by HandPoseDataset.
    Plots a 2D view (X, Y) matching the camera perspective.
    """
    
    def __init__(self):
        # Standard MediaPipe Hand topology connections (Parent Index -> Child Index)
        self.topology = [
            (0, 1), (1, 2), (2, 3), (3, 4),  # Thumb
            (0, 5), (5, 6), (6, 7), (7, 8),  # Index
            (0, 9), (9, 10), (10, 11), (11, 12), # Middle
            (0, 13), (13, 14), (14, 15), (15, 16), # Ring
            (0, 17), (17, 18), (18, 19), (19, 20), # Pinky
            
            (2, 5), # thumb to index web
            (5, 9), # index to middle web
            (9, 13), # middle to ring web
            (13, 17), # ring to pinky web
            
            (1, 5) # diagonal between thumb to index
        ]

    def show(self, hand_pose):
        """
        Renders a single hand pose.
        
        Args:
            hand_pose (torch.Tensor): Tensor of shape (1, 21, 3).
        """
        # Convert to numpy and squeeze the batch dimension
        if isinstance(hand_pose, torch.Tensor):
            landmarks = hand_pose.squeeze(0).cpu().numpy()
        else:
            landmarks = hand_pose.squeeze(0)

        # Setup the 2D plot
        fig, ax = plt.subplots(figsize=(8, 6))
        
        # Set background to black (0, 0, 0)
        fig.patch.set_facecolor('#000000')
        ax.set_facecolor('#000000')

        # We need X and Y. In your data, X is index 0, Y is index 1
        xs = landmarks[:, 0]
        ys = landmarks[:, 1]

        # Plot the skeleton (bones) in bright blue (0, 0, 255)
        # Note: Matplotlib uses RGB 0-1, so 255 is 1.0
        for parent_idx, child_idx in self.topology:
            p1 = landmarks[parent_idx]
            p2 = landmarks[child_idx]
            
            ax.plot(
                [p1[0], p2[0]], 
                [p1[1], p2[1]], 
                c=(0, 0, 1), # Bright Blue
                linewidth=2
            )

        # Plot the landmarks as slightly dimmer, transparent blue squares
        # Dimmer blue (0, 0, 0.8), alpha 0.5
        ax.scatter(
            xs, ys, 
            c=(0, 0, 0.8), 
            marker='s', # 's' for square
            alpha=0.5,  # Transparency
            s=80        # Size of the square
        )

        # Set limits to match image aspect ratio (assuming normalized 0-1 coordinates)
        ax.set_xlim(0, 1)
        ax.set_ylim(1, 0) # Invert Y to match image coordinates (0 at top)
        
        # Hide axes for a clean look
        ax.axis('off')

        plt.tight_layout()
        plt.show()

In [8]:
class HandPoseDataset(Dataset):
    def __init__(self, camera: CameraWrapper, hand_tracker: HandTracker, stop_key=' ', action_keys_list=['a', 's', 'd']):
        self.hand_poses = []
        self.hand_actions = []
        
        # this thing is used to tell where we switched from gathering to not gathering and the other way around for logging
        self.gather_flag_1 = False
        self.gather_flag_2 = self.gather_flag_1
        
        self.start_time = time.time()
        
        print(f"starting gathering loop, stop by pressing '{stop_key}'")
        
        # loop until the hault key isnt pressed
        while not keyboard.is_pressed(stop_key):
            # logging
            if self.gather_flag_1 != self.gather_flag_2:
                if self.gather_flag_1:
                    print("started gathering")
                else:
                    print("stopped gathering")

                self.gather_flag_2 = self.gather_flag_1
            
            self.gather_flag_1 = False
                
            # check which key, or if at all is pressed
            for self.key in action_keys_list:
                if keyboard.is_pressed(self.key):
                    self.current_key = self.key
                    break
            else:
                continue
            
            # if a key is indeed pressed get the hand position right now from the camera
            self.np_image = camera()
            self.hand_pose = hand_tracker(self.np_image, int((time.time() - self.start_time) * 1000))
            
            # check if no more or less than 1 hands were detected
            if len(self.hand_pose) != num_hands:
                continue
            
            self.hand_poses.append(self.hand_pose)
            self.hand_actions.append(torch.diag(torch.ones(len(action_keys_list)))[action_keys_list.index(self.current_key)]) # onehot encode currentkey's index
            
            self.gather_flag_1 = True
            
    def __len__(self):
        return len(self.hand_poses)
    
    def __getitem__(self, index):
        return self.hand_poses[index], self.hand_actions[index]

In [9]:
class PoseClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        #self.fc1 = nn.Linear(num_hands * tracking_points * tracking_dim, classifier_hidden_dim)
        #self.fc2 = nn.Linear(classifier_hidden_dim, classifier_hidden_dim)
        #self.fc3 = nn.Linear(classifier_hidden_dim, classifier_hidden_dim)
        #self.fc4 = nn.Linear(classifier_hidden_dim, num_action_classes)

        self.testfc = nn.Linear(num_hands * tracking_points * tracking_dim, num_action_classes, bias=False)

    def forward(self, x):
        x   = x.flatten(start_dim=1)
        #x   = torch.relu(self.fc1(x))
        #x   = torch.relu(self.fc2(x))
        #x   = torch.relu(self.fc3(x))
        #out = self.fc4(x)

        out = torch.sigmoid(self.testfc(x))
        # min-max normalization → [0, 1]
        lo  = out.min(dim=1, keepdim=True).values
        hi  = out.max(dim=1, keepdim=True).values
        return (out - lo) / (hi - lo + 1e-8)

    def raw(self, x):
        """Returns the unscaled linear output (pre-normalization)."""
        x = x.flatten(start_dim=1)
        return self.testfc(x)

In [10]:
class ActiveTrainer:
    def __init__(self, model, optim, camera, hand_tracker,
                 grad_accumulation_seconds=grad_accumulation_seconds,
                 num_hands=num_hands,
                 run_device=run_device,
                 action_labels=None):
        self.model = model
        self.optim = optim  # expects HumanFeedbackOptimizer

        self.camera = camera
        self.hand_tracker = hand_tracker

        self.grad_accumulation_seconds = grad_accumulation_seconds
        self.num_hands = num_hands
        self.run_device = run_device
        self.action_labels = action_labels or [str(i) for i in range(num_action_classes)]

    def _format_scores(self, scores_np):
        """Returns a list of bar-chart lines for tqdm.write(), one per class."""
        top = int(scores_np.argmax())
        lines = []
        for i, (label, s) in enumerate(zip(self.action_labels, scores_np)):
            filled = int(s * 20)
            bar = '█' * filled + '░' * (20 - filled)
            arrow = '  ←' if i == top else ''
            lines.append(f"  {label}:  {bar}  {s:.2f}{arrow}")
        return lines

    def train(self):
        self.training_running = True
        self.start_time = time.time()
        self.model.train()

        _WRITE_INTERVAL = 0.25  # seconds between score display refreshes

        window_num = 0
        while self.training_running:
            window_num += 1
            self.grad_accum_begin = time.time()
            deadline = self.grad_accum_begin + self.grad_accumulation_seconds
            last_write = 0.0

            with tqdm(
                total=self.grad_accumulation_seconds,
                desc=f"Window {window_num:>4d}",
                unit='s',
                bar_format='{desc}: {percentage:3.0f}%|{bar}| {n:.1f}/{total:.0f}s',
                ncols=60,
                leave=True,
            ) as pbar:
                while time.time() < deadline:
                    self.np_image = self.camera()
                    self.hand_pose = self.hand_tracker(
                        self.np_image, int((time.time() - self.start_time) * 1000)
                    )

                    if self.hand_pose.size(0) != self.num_hands:
                        continue

                    offsets = to_bone_offsets(self.hand_pose)
                    self.prediction = self.model(offsets.to(self.run_device))
                    loss_val = self.optim.accumulate(self.prediction)

                    # sync bar to real elapsed time
                    pbar.n = min(time.time() - self.grad_accum_begin, self.grad_accumulation_seconds)
                    pbar.refresh()

                    # rate-limited score display so the output doesn't scroll
                    now = time.time()
                    if now - last_write >= _WRITE_INTERVAL:
                        scores = self.prediction.detach().cpu().numpy()[0]
                        lines = self._format_scores(scores)
                        fb = f"fb: {loss_val:+.3f}" if loss_val is not None else "fb: dead zone"
                        lines.append(f"  {fb}")
                        clear_output()
                        tqdm.write('\n'.join(lines))
                        last_write = now

            self.optim.step()

In [11]:
pose_classifier = PoseClassifier().to(run_device)

In [12]:
hand_tracker = HandTracker(model_path="hand_landmarker.task")
camera = CameraWrapper(1)

In [13]:
# optimizer is instantiated in the next cell alongside HumanFeedbackOptimizer

In [14]:
base_optim = optim(pose_classifier.parameters(), lr=lr)
feedback_optim = HumanFeedbackOptimizer(base_optim, screen_height=target_monitor_y_res)

In [15]:
trainer = ActiveTrainer(pose_classifier, feedback_optim, camera, hand_tracker, action_labels=['a', 's', 'd'])

In [ ]:
# ── TRAINING ─────────────────────────────────────────────────────────────────
# Move mouse to give feedback each accumulation window (default: every 5 s):
#   Top of screen    → feedback ≈ +1  →  reinforce current prediction
#   Bottom of screen → feedback ≈ -1  →  punish current prediction
#   Middle dead zone → feedback ≈  0  →  skip update
#
# Stop: Kernel > Interrupt (or Ctrl+C in terminal)
# ─────────────────────────────────────────────────────────────────────────────
trainer.train()

Window    4:  93%|███████████████████████████████▋  | 4.7/5s

  a:  ░░░░░░░░░░░░░░░░░░░░  0.00
  s:  ███████████████████░  1.00  ←
  fb: -0.160


Window    4:  93%|███████████████████████████████▋  | 4.7/5s
Window    5:   0%|                                  | 0.0/5s

In [ ]:
# ── INFERENCE ────────────────────────────────────────────────────────────────
# Runs until you press 'q'. Run all setup cells above before this.
# ─────────────────────────────────────────────────────────────────────────────
pose_classifier.eval()
hand_tracker.reset()
start_time = time.time()

print("Running inference — press 'q' to stop\n")

with torch.no_grad():
    while not keyboard.is_pressed('q'):
        np_image  = camera()
        hand_pose = hand_tracker(np_image, int((time.time() - start_time) * 1000))

        if hand_pose.size(0) != num_hands:
            continue

        offsets       = to_bone_offsets(hand_pose)
        scores        = pose_classifier(offsets.to(run_device))[0].cpu().numpy()
        predicted_idx = int(scores.argmax())

        if press_buttons:
            pyautogui.press(action_labels[predicted_idx])

        clear_output(wait=True)
        print(f"Predicted: [ {action_labels[predicted_idx]} ]\n")
        for i, (label, s) in enumerate(zip(action_labels, scores)):
            filled = int(s * 20)
            bar = '█' * filled + '░' * (20 - filled)
            marker = '  ←' if i == predicted_idx else ''
            print(f"  {label}:  {bar}  {s:.2f}{marker}")

print("\nStopped.")